# nn-parameter-wrap — ex2: Parameter vs buffer — pick the right registration

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `nn-parameter-wrap`. Running the final beacon cell reports progress against the `PyTorch: nn.Parameter` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: nn.Parameter` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`nn-parameter-wrap`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "nn-parameter-wrap"
DD_SUBTOPIC = "PyTorch: nn.Parameter"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `nn.Parameter` — quick refresher

`nn.Parameter(tensor)` is a tensor subclass with one job: when assigned as an attribute of an `nn.Module`, it gets auto-registered in the module's `_parameters` dict so it shows up in `.parameters()`, `.state_dict()`, and gets moved by `.to(device)` / `.cuda()`.

**Parameter vs raw tensor.** `self.w = torch.randn(3)` — invisible to the optimizer. `self.w = nn.Parameter(torch.randn(3))` — included.

**Parameter vs buffer.** Both round-trip through `state_dict()`. Only Parameters are trainable (`.requires_grad=True` by default and included in `.parameters()`). Buffers are for non-learnable state — running stats, position encodings, attention masks. Register with `self.register_buffer('running_mean', torch.zeros(C))`.

**Gotcha — default dtype.** `nn.Parameter(torch.tensor([1, 2, 3]))` creates an `int64` parameter, which optimizers reject. Always pass a float tensor or call `.float()` first.

### Exercise 2 — Parameter vs buffer — pick the right registration

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Build a BatchNorm-style module that registers a learnable weight as nn.Parameter and a running statistic as a buffer, and verify the two registries differ correctly.
> Keywords: register_buffer, nn.Parameter, state_dict, requires_grad
> ```

**KCs targeted:** `parameter-vs-buffer`, `parameter-wrap-tensor`

Implement `RunningScaler` — a stripped-down stand-in for BatchNorm that exercises the Parameter-vs-buffer choice:

1. `class RunningScaler(t.nn.Module)` with `__init__(self, num_features)`:
   - Call `super().__init__()` first.
   - Register `weight` as a learnable `nn.Parameter` initialized to `t.ones(num_features)`.
   - Register `running_mean` as a BUFFER (NOT a Parameter) initialized to `t.zeros(num_features)`, using `self.register_buffer('running_mean', t.zeros(num_features))`.
2. `forward(self, x: Tensor) -> Tensor`:
   - `x` has shape `(B, num_features)`.
   - During training (`self.training is True`): update `self.running_mean` with the batch mean using EMA with momentum `0.1`:
     `self.running_mean = 0.9 * self.running_mean + 0.1 * x.mean(dim=0)`
     Wrap the update in `with t.no_grad():` so it doesn't pollute the autograd graph.
   - Return `(x - self.running_mean) * self.weight` (broadcasts over the batch dim).

Return an instance from `ex2_build_running_scaler(num_features)`.

The test confirms `weight` lives in `parameters()`, `running_mean` lives in `buffers()` (NOT in `parameters()`), and BOTH round-trip via `state_dict()`.

In [ ]:
def ex2_build_running_scaler(num_features: int):
    class RunningScaler(t.nn.Module):
        def __init__(self, num_features):
            super().__init__()
            self.weight = t.nn.Parameter(t.ones(num_features))
            self.register_buffer('running_mean', t.zeros(num_features))
        def forward(self, x: Tensor) -> Tensor:
            if self.training:
                with t.no_grad():
                    batch_mean = x.mean(dim=0)
                    self.running_mean = 0.9 * self.running_mean + 0.1 * batch_mean
            return (x - self.running_mean) * self.weight
    return RunningScaler(num_features)


<details><summary>Solution</summary>

```python
def ex2_build_running_scaler(num_features: int):
    class RunningScaler(t.nn.Module):
        def __init__(self, num_features):
            super().__init__()
            self.weight = t.nn.Parameter(t.ones(num_features))
            self.register_buffer('running_mean', t.zeros(num_features))
        def forward(self, x: Tensor) -> Tensor:
            if self.training:
                with t.no_grad():
                    batch_mean = x.mean(dim=0)
                    self.running_mean = 0.9 * self.running_mean + 0.1 * batch_mean
            return (x - self.running_mean) * self.weight
    return RunningScaler(num_features)
```

**Three registries, not two.** `nn.Module` separates state into three internal dicts:
- `_parameters` — learnable, in `.parameters()`, in `state_dict()`, `requires_grad=True` by default.
- `_buffers` — non-learnable, NOT in `.parameters()`, in `state_dict()`, `requires_grad=False`.
- `_modules` — child modules, recursively contribute their own parameters + buffers.

**Why buffers and not raw tensors.** Buffers move with `.cuda()`, round-trip via `state_dict()`, get included in `.to(device)`. A raw tensor attribute survives none of that.

**Why the `with t.no_grad():` guard.** Without it, the EMA update `0.9 * self.running_mean + 0.1 * batch_mean` would build an autograd graph through every training step, eventually OOMing or breaking `loss.backward()`. Buffers are non-learnable; their updates must not participate in autograd.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()